In [ ]:
from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import bvdpdf
from time import sleep
import os

print("Running KY CIMA Web Scraping Tool v.1.0")

now=datetime.datetime.now()
filename= 'KY CIMA SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])

scriptfolder = os.path.dirname(os.path.abspath(__file__))
os.chdir(scriptfolder)
tempfolder = os.path.join(scriptfolder, 'tempfolder')

if os.path.exists(tempfolder):
    for rem_file in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem_file))
else:
    os.mkdir(tempfolder)

chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder}
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


def fix_year(year):
    if int(year)>1900:
        return year
    elif int(year)>100:
        error_message = f'Invalid year: {year}'
        raise Exception(error_message)
    elif int(year)>40:
        return f'19{year}'
    else:
        return f'20{year}'

def month_to_num(month):
    months = {
        'jan': '01',
        'feb': '02',
        'mar': '03',
        'apr': '04',
         'may': '05',
         'jun': '06',
         'jul': '07',
         'aug': '08',
         'sep': '09',
         'oct': '10',
         'nov': '11',
         'dec': '12'
        }
    try:
        return months[month.strip()[:3].lower()]
    except:
        raise ValueError('Not a month')

def fix_dfs(dfs_list):
    fixed_dfs = list()
    for df_temporal in  dfs_list:
        df_temporal.loc[-1] = df_temporal.columns.tolist()  # adding original columns as row 1
        df_temporal.index = df_temporal.index + 1  # shifting index
        df_temporal.columns = [numb for numb in range(len(df_temporal.columns))]  # re-assigning columns
        df_temporal.sort_index(inplace=True)
        fixed_dfs.append(df_temporal.reset_index(drop=True))
    return fixed_dfs
    
def join_dfs(dfs_list):
    fixed_dfs = fix_dfs(dfs_list)
    joined_df = pd.concat(fixed_dfs).reset_index(drop=True)
    name_headers = ['Entity Name', 'Institution Name', 'License Name', 'Mutual Fund Name', 'Mutual Fund Administrator Name']
    for row_nr in range(joined_df.shape[0]):
        row_cells = [str(ele) for ele in joined_df.iloc[row_nr,:].tolist()]
        #print(row_nr, row_cells)
        name_bool = [True if cell in name_headers  else False for cell in row_cells]
        if any(name_bool):
            name_header_value = row_cells[name_bool.index(True)]
            original_cols = [c_n if len(c_n)>0 else '_' for c_n in row_cells]
            new_cols = list() #cant use constructor direclty as they may be duplicate col names
            for i, new_col in enumerate(original_cols):
                if original_cols.count(new_col)>1:
                    new_cols.append('{}_{}'.format(new_col, original_cols[:i+1].count(new_col)))
                else:
                    new_cols.append(new_col)
            joined_df.columns = new_cols
            break
    else:
        raise Exception('No name column found after iterating all rows')
    joined_df =joined_df[row_nr+1:].reset_index(drop=True)
    joined_df[new_cols] = joined_df[new_cols].astype(str)
    joined_df = joined_df.replace(name_header_value, np.nan, regex=True)#replacing name_header from 2nd+pages with with np.nan (NaN)
    joined_df = joined_df.replace(r'^\s*$', np.nan, regex=True) #replacing White Space with NaN
    joined_df = joined_df.replace(r'^[0-9]$', np.nan, regex=True) #replacing cells with a single number with a NaN
    joined_df = joined_df.replace(r'^Total.*:$', np.nan, regex=True) #replacing 'Total Funds:' and/or 'Total Entities:' NaN
    joined_df = joined_df.dropna(axis = 0, how = 'all', subset = joined_df.columns.tolist())#deleting rows full of NaN
    joined_df = joined_df.dropna(axis = 0, how = 'all', subset = [name_header_value])#delete rows where name is NaN
    return joined_df.reset_index(drop=True)

def fill_sqldict(dfs_list):
    #global sqldict#check if this doesnt replace and restarts the dict
    joined_df = join_dfs(dfs_list)
    name_headers = ['Entity Name', 'Institution Name', 'License Name', 'Mutual Fund Name', 'Mutual Fund Administrator Name']
    entity_col_name = None
    lic_col_name = None
    date_col_name = None
    for col_name in joined_df.columns.tolist():
        if entity_col_name is None and col_name in name_headers:
            entity_col_name = col_name
            joined_df.loc[:, entity_col_name].replace('', np.nan, inplace=True)
            joined_df = joined_df.dropna(axis = 0, how = 'all', subset = [entity_col_name]).reset_index(drop=True)
        elif lic_col_name is None and col_name in ['Licence #' ,'Licence Number', 'License Number', 'Licence#', 'License #']:
            lic_col_name = col_name###, inplace=True
            joined_df.loc[:, lic_col_name].replace('', np.nan, inplace=True)
            joined_df = joined_df.dropna(axis = 0, how = 'all', subset = [lic_col_name]).reset_index(drop=True)
        elif date_col_name is None and col_name in ['Date Licenced', 'Recognised']: #mm/dd/yyyy to yyyy-mm-dd
            date_col_name = col_name
            joined_df.loc[:, date_col_name].replace('', np.nan, inplace=True)
            joined_df = joined_df.dropna(axis = 0, how = 'all', subset = [date_col_name]).reset_index(drop=True)
            print(date_col_name, joined_df[date_col_name].tolist())
    sqldict['Name'].extend(joined_df[entity_col_name].tolist())
    if lic_col_name:
        sqldict['InternalID_1_type'].extend([lic_col_name for ele in joined_df[lic_col_name].tolist()])
        sqldict['InternalID_1'].extend(joined_df[lic_col_name].tolist())
    if date_col_name:
        dates = [str(ele).replace('-', '/').split('/') for ele in joined_df[date_col_name].tolist()]
        #dates = [ele.split('-') if '-' in ele else ele.split('/') for ele in dates]
        #print(dates)
        print(dates[0], dates[0][1].isalpha())
        if dates[0][1].isalpha():
            sqldict['RegulationDate'].extend(['{}-{}-{}'.format(fix_year(ele[2]), month_to_num(ele[1]), ele[0]) if len(ele) >1 else '' for ele in dates])
        else:
            sqldict['RegulationDate'].extend(['{}-{}-{}'.format(fix_year(ele[2]), ele[0], ele[1]) if len(ele) >1 else '' for ele in dates])
    return joined_df


#always in list of entities:
regdict={'KY CIMA 1': ['https://www.cima.ky/banking-statistics', 'List of Category A Banks Q'] ,
	 	 'KY CIMA 2': ['https://www.cima.ky/banking-statistics', 'List of Category B Banks Q'] ,
	 	 'KY CIMA 3': ['https://www.cima.ky/money-services-statistics','List of Money Services Business Providers Q'] ,
	 	 'KY CIMA 4': ['https://www.cima.ky/trusts-statistics', 'List of (unrestricted) Trust Companies licensed with the Cayman Islands Monetary Authority'] ,
	 	 'KY CIMA 5': ['https://www.cima.ky/trusts-statistics', 'List of Restricted Trust Companies Licensed with the Cayman Islands Monetary Authority'] ,
	 	 'KY CIMA 6': ['https://www.cima.ky/trusts-statistics', 'List of Nominee Companies licensed with the Cayman Islands Monetary Authority and supervised by the Fiduciary Services Division'] ,
	 	 'KY CIMA 7': ['https://www.cima.ky/trusts-statistics', 'List of Controlled Subsidiaries registered with the Cayman Islands Monetary Authority'] ,
	 	 'KY CIMA 8': ['https://www.cima.ky/trusts-statistics', 'List of PTCs Registered with the Cayman Islands Monetary Authority'] ,
	 	 'KY CIMA 9': ['https://www.cima.ky/corporate-services-statistics', 'List of Company Managers Licensed with the Cayman Islands Monetary Authority'] ,
	 	 'KY CIMA 10': ['https://www.cima.ky/corporate-services-statistics', 'List of Corporate Service Providers Licensed with the Cayman Islands Monetary Authority'] ,
	 	 'KY CIMA 11': ['https://www.cima.ky/insurance-statistics', 'Full List of Insurance Licensees by Licence Type Q'] ,
	 	 'KY CIMA 12': ['https://www.cima.ky/investment-statistics', 'Quarterly List of all Mutual Funds registered and licensed with the Cayman Islands Monetary Authority'] ,
	 	 'KY CIMA 13': ['https://www.cima.ky/investment-statistics', 'Quarterly List of all Mutual Fund Administrators licensed with the Cayman Islands Monetary Authority']}


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

processdate=now.strftime('%Y-%m-%d')

#Does  complete_pdfs? exists?
for key, c_pdf in complete_pdfs.items():
    print(key)
    joined_df = fill_sqldict(c_pdf)
    rows = joined_df.shape[0]
    print('rows:', rows)
    sqldict['ListProcessDate'].extend([processdate for ele in range(rows)])
    sqldict['Cntry'].extend(['KY' for ele in range(rows)])
    sqldict['RegCtry'].extend(['KY' for ele in range(rows)])
    sqldict['RegCode'].extend(['CIMA' for ele in range(rows)])
    sqldict['ListCode'].extend([key.split()[-1] for ele in range(rows)])
    sqldict['RegulationType'].extend(['Regulated' for ele in range(rows)])
    for k_ in sqldict:
        if len(sqldict['Name'])>len(sqldict[k_]):
            sqldict[k_].extend(['' for ele in range(rows)])



for reg in regdict:
    print('Working with {}'.format(reg))
    driver.get('about:blank')
    driver.get(regdict[reg][0])
    sleep(4)
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    pdf_urls = soup.find_all('a', {'data-viewid': True, 'href': True, 'title': True} )
    pdf_href = [ele['href'] for ele in pdf_urls if regdict[reg][1] in ele['title']  and ele['href'].lower().endswith('.pdf')][0]
    driver.get(pdf_href)
    for times in range(50):
        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele])==0 and len(os.listdir(tempfolder))>0:
            print('Waiting for file to download')
            sleep(2)
        else:
            break
    else:
        Exception('Failed to download PDF file')
    pdf_path = os.path.join(tempfolder, os.listdir(tempfolder)[0])
    #tables = tabula.read_pdf(pdf_path, pages='all', multiple_tables=True)
    tables = bvdpdf.get_tables(pdf_path, header=None)
    os.remove(pdf_path)
    joined_df = fill_sqldict(tables)
    rows = joined_df.shape[0]
    sqldict['ListProcessDate'].extend([processdate for ele in range(rows)])
    sqldict['Cntry'].extend(['KY' for ele in range(rows)])
    sqldict['RegCtry'].extend(['KY' for ele in range(rows)])
    sqldict['RegCode'].extend(['CIMA' for ele in range(rows)])
    sqldict['ListCode'].extend([key.split()[-1] for ele in range(rows)])
    sqldict['RegulationType'].extend(['Regulated' for ele in range(rows)])
    for key in sqldict:
        if len(sqldict['Name'])>len(sqldict[key]):
            sqldict[key].extend(['' for ele in range(rows)])

df=pd.DataFrame(sqldict)
writer = ExcelWriter(filename)
df.to_excel(writer, 'SQL Ready', index=False)
writer.save()
writer.close()

sleep(3)

driver.quit()
    
    
    
    